# Pixeltable as a Multimodal Feature Store for Ray Train

An end-to-end ML pipeline using Pixeltable and Ray:

| Step | Tool | What happens |
|------|------|--------------|
| **Ingest** | Pixeltable | Store raw images + text descriptions |
| **Preprocess** | Pixeltable computed columns | Extract image stats + text features automatically on insert |
| **Train** | Ray Train + PixeltableDatasource | Distributed MLP training, data streamed from Pixeltable |
| **Deploy** | Pixeltable UDF | Trained model scores every new row on insert — no separate prediction pipeline |
| **Batch re-score** | PixeltableDatasink | Ray writes enriched results back to a Pixeltable analytics table |

No external API keys required. Feature extraction runs locally with PIL and NumPy.

In [1]:
import logging
import warnings
logging.getLogger('ray').setLevel(logging.ERROR)
logging.getLogger('pixeltable').setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

import os
import tempfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import pixeltable as pxt
from pixeltable.io.ray import PixeltableDatasource, PixeltableDatasink
import ray
import ray.data
from ray import train
from ray.train import ScalingConfig, Checkpoint
from ray.train.torch import TorchTrainer

ray.init(ignore_reinit_error=True, logging_level=logging.ERROR, log_to_driver=False)

pxt.drop_dir('catalog', force=True, if_not_exists='ignore')
pxt.create_dir('catalog')
print('Setup complete.')

Connected to Pixeltable database at: postgresql+psycopg://postgres:@/pixeltable?host=/Users/pjlb/.pixeltable/pgdata
Found an existing Pixeltable dashboard at: http://localhost:22089
Created directory 'catalog'.
Setup complete.


## 1. Ingest — multimodal product catalog

Each record has an **image** (URL), a text **description**, and a binary **label**:
- `1` = premium listing: vivid image + detailed, rich description
- `0` = basic listing: same images with terse, minimal descriptions

Only raw data is stored here. All feature extraction is declarative — defined in the next section.

In [2]:
BASE = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images/'
IMG_IDS = [
    '000000000001', '000000000016', '000000000019', '000000000025',
    '000000000030', '000000000034', '000000000036', '000000000042',
    '000000000049', '000000000061', '000000000090', '000000000106',
    '000000000139', '000000000285', '000000000776', '000000000885',
]

PREMIUM = [
    'Vibrant farmers market bursting with colorful fresh produce, lively vendors, and warm morning light.',
    'Artisan kitchen counter styled with rustic ceramics, fresh herbs, and warm overhead lighting.',
    'Bustling outdoor plaza with street art, diverse crowds, and golden afternoon sunlight.',
    'Scenic waterfront at dusk: sailboats, reflections, and a dramatic orange-to-blue gradient sky.',
    'Gourmet plating on matte slate with microgreens, edible flowers, and a rich reduction sauce.',
    'Curated rooftop terrace with potted plants, string lights, and a sweeping city skyline view.',
    'Wildlife photography: bird of prey mid-flight against a crisp mountain backdrop.',
    'Heritage bakery window with hand-lettered chalk signs, golden pastries, and warm ambience.',
    'Sunset yoga on a cliffside overlooking the ocean, silhouetted against coral and purple tones.',
    'Modern gym with sleek machines, motivational murals, and dynamic directional lighting.',
]
BASIC = [
    'Items on a surface.',
    'A room.',
    'Street.',
    'Water.',
    'Food.',
    'Building.',
    'Animal.',
    'Objects.',
    'Scene.',
    'Interior.',
]

rows = (
    [{'image': BASE + IMG_IDS[i % len(IMG_IDS)] + '.jpg',
      'description': PREMIUM[i], 'label': 1} for i in range(10)]
    + [{'image': BASE + IMG_IDS[(i + 10) % len(IMG_IDS)] + '.jpg',
        'description': BASIC[i], 'label': 0} for i in range(10)]
)

catalog = pxt.create_table('catalog.listings', {
    'image': pxt.Image,
    'description': pxt.String,
    'label': pxt.Int,
})
catalog.insert(rows)
print(f'Ingested {catalog.count()} rows')
catalog.select(catalog.description, catalog.label).limit(4).collect()

description,label
"Vibrant farmers market bursting with colorful fresh produce, lively vendors, and warm morning light.",1
"Artisan kitchen counter styled with rustic ceramics, fresh herbs, and warm overhead lighting.",1
"Bustling outdoor plaza with street art, diverse crowds, and golden afternoon sunlight.",1
"Scenic waterfront at dusk: sailboats, reflections, and a dramatic orange-to-blue gradient sky.",1


## 2. Preprocess — declarative feature pipeline via computed columns

Two UDFs define the feature extraction logic:

- **`image_features`** — 6-dim vector: mean & std of each RGB channel from a 64×64 thumbnail
- **`text_features`** — 4-dim vector: word count, char count, average word length, lexical diversity

Pixeltable runs these **automatically** on every existing and future row. The results are stored and never recomputed unless the source data changes.

In [3]:
@pxt.udf
def image_features(img: pxt.Image) -> pxt.Array[(6,), pxt.Float]:
    """Mean and std of R, G, B channels from a 64×64 thumbnail."""
    arr = np.array(img.convert('RGB').resize((64, 64)), dtype=np.float32) / 255.0
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    return np.array([r.mean(), g.mean(), b.mean(), r.std(), g.std(), b.std()], dtype=np.float32)


@pxt.udf
def text_features(text: str) -> pxt.Array[(4,), pxt.Float]:
    """Word count, char count, avg word length, lexical diversity — all normalised to [0, 1]."""
    words = text.lower().split()
    n = len(words) or 1
    return np.array([
        min(n / 30.0, 1.0),
        min(len(text) / 150.0, 1.0),
        min((sum(len(w) for w in words) / n) / 8.0, 1.0),
        len(set(words)) / n,
    ], dtype=np.float32)


catalog.add_computed_column(img_feats=image_features(catalog.image))
catalog.add_computed_column(txt_feats=text_features(catalog.description))

print('Feature columns computed. Sample:')
catalog.select(catalog.label, catalog.img_feats, catalog.txt_feats).limit(3).collect()

label,img_feats,txt_feats
1,[0.331 0.343 0.274 0.223 0.215 0.225],[0.467 0.667 0.777 1. ]
1,[0.448 0.432 0.417 0.145 0.13 0.132],[0.433 0.62 0.779 1. ]
1,[0.393 0.464 0.309 0.221 0.177 0.303],[0.4 0.573 0.781 1. ]


## 3. Inspect — query features directly in Pixeltable

In [4]:
import pandas as pd

rs = catalog.select(catalog.label, catalog.img_feats, catalog.txt_feats).collect()
df = pd.DataFrame({
    'label':   rs['label'],
    'r_mean':  [v[0] for v in rs['img_feats']],
    'g_mean':  [v[1] for v in rs['img_feats']],
    'b_mean':  [v[2] for v in rs['img_feats']],
    'word_cnt':[v[0] for v in rs['txt_feats']],
    'char_cnt':[v[1] for v in rs['txt_feats']],
    'lex_div': [v[3] for v in rs['txt_feats']],
})

print('Mean feature values by label:')
print(df.groupby('label')[['word_cnt','char_cnt','lex_div','r_mean','g_mean','b_mean']].mean().round(3))


Mean feature values by label:
       word_cnt  char_cnt  lex_div  r_mean  g_mean  b_mean
label                                                     
0         0.047     0.055      1.0   0.447   0.434   0.331
1         0.423     0.604      1.0   0.475   0.477   0.396


## 4. Train — Ray Train streams data from Pixeltable

The dataset is passed to `TorchTrainer` via `datasets=`. Inside each worker, `train.get_dataset_shard()` returns the worker's slice, and `iter_batches(batch_format='numpy')` streams it without any pandas conversion.

Array columns (`img_feats`, `txt_feats`) arrive as object-dtype numpy arrays of shape `(B,)` where each element is a 1-D array — `np.stack()` reshapes them to `(B, D)` before converting to tensors.

In [5]:
MODEL_PATH = '/tmp/pxt_listing_classifier.pt'


def build_model() -> nn.Module:
    return nn.Sequential(
        nn.Linear(10, 32), nn.ReLU(),
        nn.Linear(32, 16), nn.ReLU(),
        nn.Linear(16, 1),  nn.Sigmoid(),
    )


def train_loop(config: dict) -> None:
    """Runs on each Ray Train worker. No pandas — pure Ray Data streaming."""
    import os, tempfile
    import numpy as np
    import torch
    import torch.nn as nn
    from pixeltable.io.ray import PixeltableDatasource
    from ray import train
    from ray.train import Checkpoint
    # ── Get this worker's data shard ──────────────────────────────────────
    shard = train.get_dataset_shard('train')

    model     = train.torch.prepare_model(build_model())
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    loss_fn   = nn.BCELoss()

    for epoch in range(80):
        model.train()
        epoch_loss = 0.0
        n_batches  = 0

        # iter_batches streams without loading the entire shard into memory
        for batch in shard.iter_batches(batch_size=8, batch_format='numpy'):
            # Array columns are object-arrays of 1-D arrays → stack to (B, D)
            img = torch.tensor(np.stack(batch['img_feats']), dtype=torch.float32)  # (B, 6)
            txt = torch.tensor(np.stack(batch['txt_feats']), dtype=torch.float32)  # (B, 4)
            y   = torch.tensor(batch['label'],               dtype=torch.float32).unsqueeze(1)
            x   = torch.cat([img, txt], dim=1)                                     # (B, 10)

            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches  += 1

        # Save checkpoint — required for result.metrics to be populated in Ray Train v2
        inner = model.module if hasattr(model, 'module') else model
        with tempfile.TemporaryDirectory() as tmpdir:
            torch.save(inner.state_dict(), os.path.join(tmpdir, 'model.pt'))
            train.report(
                {'epoch': epoch + 1, 'loss': round(epoch_loss / max(n_batches, 1), 4)},
                checkpoint=Checkpoint.from_directory(tmpdir),
            )


# ── Build the Ray Dataset once on the driver ──────────────────────────────
ds = ray.data.read_datasource(
    PixeltableDatasource('catalog.listings', columns=['img_feats', 'txt_feats', 'label']),
    override_num_blocks=4,
)

trainer = TorchTrainer(
    train_loop,
    datasets={'train': ds},           # passed to workers; Ray shards automatically
    scaling_config=ScalingConfig(num_workers=2, use_gpu=False),
)
result = trainer.fit()

# Persist best checkpoint for the inference UDF
import shutil
ckpt_dir = result.checkpoint.to_directory()
shutil.copy(os.path.join(ckpt_dir, 'model.pt'), MODEL_PATH)

print(f'Training complete.  Final loss: {result.metrics["loss"]}')

Training complete.  Final loss: 0.0181


## 5. Deploy — register the model as a Pixeltable UDF

Wrapping the model in `@pxt.udf` and adding it as a **computed column** means:
- All existing rows are scored immediately
- Every future `insert()` triggers inference automatically — the feature pipeline and model are one declarative chain

In [6]:
_model_cache: dict = {}

def _get_classifier() -> nn.Module:
    """Lazy-load the model once and cache it for the process lifetime."""
    if 'model' not in _model_cache:
        m = build_model()
        m.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
        m.eval()
        _model_cache['model'] = m
    return _model_cache['model']


@pxt.udf
def predict_quality(
    img_feats: pxt.Array[(6,), pxt.Float],
    txt_feats: pxt.Array[(4,), pxt.Float],
) -> pxt.Float:
    """Return probability [0, 1] that the listing is premium quality."""
    model = _get_classifier()
    x = torch.tensor(
        np.concatenate([img_feats, txt_feats]).astype(np.float32)
    ).unsqueeze(0)
    with torch.no_grad():
        return float(model(x).item())


catalog.add_computed_column(
    quality_score=predict_quality(catalog.img_feats, catalog.txt_feats)
)

print('Model deployed. Scores on training data:')
catalog.select(
    catalog.label, catalog.description, catalog.quality_score
).order_by(catalog.quality_score, asc=False).limit(6).collect()

label,description,quality_score
1,"Sunset yoga on a cliffside overlooking the ocean, silhouetted against coral and purple tones.",0.991
1,"Vibrant farmers market bursting with colorful fresh produce, lively vendors, and warm morning light.",0.991
1,"Gourmet plating on matte slate with microgreens, edible flowers, and a rich reduction sauce.",0.991
1,"Artisan kitchen counter styled with rustic ceramics, fresh herbs, and warm overhead lighting.",0.985
1,"Curated rooftop terrace with potted plants, string lights, and a sweeping city skyline view.",0.983
1,"Heritage bakery window with hand-lettered chalk signs, golden pastries, and warm ambience.",0.982


## 6. Evaluate

In [7]:
import numpy as np
import pandas as pd

rs = catalog.select(catalog.label, catalog.quality_score).collect()
scores    = np.array(list(rs['quality_score']), dtype=float)
labels    = np.array(list(rs['label']),         dtype=int)
predicted = (scores > 0.5).astype(int)
acc = (predicted == labels).mean()
print(f'Accuracy: {acc:.0%}  ({int(acc * len(labels))}/{len(labels)})')
results_df = pd.DataFrame({'label': labels, 'quality_score': scores})
print('\nScore distribution by class:')
print(results_df.groupby('label')['quality_score'].describe().round(3))
results_df['predicted'] = (results_df['quality_score'] > 0.5).astype(int)
acc = (results_df['predicted'] == results_df['label']).mean()
print(f'Accuracy: {acc:.0%}  ({int(acc * len(results_df))}/{len(results_df)})')
print('\nScore distribution by class:')
print(results_df.groupby('label')['quality_score'].describe().round(3))

Accuracy: 100%  (20/20)

Score distribution by class:
       count   mean    std    min    25%    50%    75%    max
label                                                        
0       10.0  0.031  0.032  0.015  0.017  0.019  0.021  0.120
1       10.0  0.980  0.011  0.964  0.971  0.982  0.989  0.991
Accuracy: 100%  (20/20)

Score distribution by class:
       count   mean    std    min    25%    50%    75%    max
label                                                        
0       10.0  0.031  0.032  0.015  0.017  0.019  0.021  0.120
1       10.0  0.980  0.011  0.964  0.971  0.982  0.989  0.991


## 7. New data — fully automated inference on insert

Insert raw rows. Pixeltable automatically runs the full chain:
`image_features` → `text_features` → `predict_quality`

No explicit call to a prediction function needed.

In [8]:
new_listings = [
    {
        'image': BASE + '000000000036.jpg',
        'description': "Award-winning chef's table: seasonal tasting menu, impeccable plating, curated wine pairings in an intimate setting.",
        'label': 1,
    },
    {
        'image': BASE + '000000000049.jpg',
        'description': 'Thing.',
        'label': 0,
    },
    {
        'image': BASE + '000000000090.jpg',
        'description': 'Handcrafted artisan coffee bar featuring single-origin roasts, pour-over stations, and Scandinavian interior design.',
        'label': 1,
    },
    {
        'image': BASE + '000000000106.jpg',
        'description': 'Place.',
        'label': 0,
    },
]

catalog.insert(new_listings)

# Retrieve only the newly inserted rows
all_rs = catalog.select(
    catalog.label, catalog.description, catalog.quality_score
).collect()
all_rs = catalog.select(
    catalog.label, catalog.description, catalog.quality_score
).collect()
new_df = pd.DataFrame({
    'label':         list(all_rs['label'])[-4:],
    'description':   list(all_rs['description'])[-4:],
    'quality_score': list(all_rs['quality_score'])[-4:],
}).reset_index(drop=True)
new_df['predicted'] = (new_df['quality_score'] > 0.5).astype(int)
new_df['correct']   = new_df['predicted'] == new_df['label']
print('Inference on 4 new listings (auto-scored on insert):')
new_df[['label', 'predicted', 'quality_score', 'correct', 'description']]

Inference on 4 new listings (auto-scored on insert):


,label,predicted,quality_score,correct,description
0,1,1,0.997970,True,Award-winning chef's table: seasonal tasting m...
1,0,0,0.024467,True,Thing.
2,1,1,0.995997,True,Handcrafted artisan coffee bar featuring singl...
3,0,0,0.023404,True,Place.


## 8. Batch re-score — Ray reads from Pixeltable, enriches, writes back

For large-scale scheduled jobs (e.g., nightly re-ranking), Ray can read all scored records from Pixeltable, apply distributed post-processing, and write results to a separate analytics table — all without leaving the Ray + Pixeltable ecosystem.

In [9]:
scored_tbl = pxt.create_table('catalog.scored', {
    'label': pxt.Int,
    'quality_score': pxt.Float,
    'tier': pxt.String,
})

# Read all scored listings from Pixeltable as a Ray Dataset
scored_ds = ray.data.read_datasource(
    PixeltableDatasource('catalog.listings', columns=['label', 'quality_score']),
    override_num_blocks=2,
)

# Distributed post-processing: assign tiers
def assign_tier(row: dict) -> dict:
    s = row['quality_score']
    row['tier'] = 'premium' if s >= 0.75 else ('standard' if s >= 0.4 else 'basic')
    return row

scored_ds.map(assign_tier).write_datasink(PixeltableDatasink('catalog.scored'))

print('Batch re-score complete. Tier distribution:')
sr = scored_tbl.select(scored_tbl.tier, scored_tbl.quality_score).collect()
pd.DataFrame({'tier': list(sr['tier']), 'quality_score': list(sr['quality_score'])}) \
    .groupby('tier')['quality_score'].agg(['count', 'mean']).round(3)

,count,mean
tier,,
basic,12,0.029
premium,12,0.982


## Cleanup

In [10]:
pxt.drop_dir('catalog', force=True)
ray.shutdown()
print('Done.')

Done.
